# 02. Streamlit 앱/배포 번들 생성 및 GitHub push

이 노트북은 다음 작업을 합니다.

1. `app/streamlit_app.py` 생성/갱신
2. `requirements-ec2.txt`, Streamlit config, systemd service 파일 생성
3. Drive의 학습 결과 CSV/PNG를 `docs/assets/results`로 복사
4. Drive의 best checkpoint와 sample pool을 `deploy_bundle.zip`으로 묶음
5. README에 결과 섹션 삽입
6. `DEFECTVISION_GH_TOKEN` 환경변수로 GitHub push

In [1]:
# ===== 공통 환경 설정: Colab + Drive + GitHub repo + 안전한 import 경로 =====
# 이 셀은 모든 노트북에서 가장 먼저 실행하세요.
# 핵심 원칙:
# - requirements.txt 전체 설치 금지
# - numpy / pandas / torch / torchvision / opencv / scikit-learn 강제 재설치 금지
# - 데이터, checkpoint, deploy bundle은 Google Drive에 저장
# - GitHub에는 코드, README, 작은 시각화 파일, demo sample 이미지만 업로드

import os
import sys
import json
import shutil
import subprocess
from pathlib import Path
from getpass import getpass

GITHUB_REPO_URL = "https://github.com/wnstjq0915/DefectVision-AD-Proj.git"
GITHUB_USERNAME = "wnstjq0915"
GITHUB_EMAIL = "wnstjq0915@gmail.com"
PROJECT_DIR = Path("/content/DefectVision-AD-Proj")
PROJECT_NAME = "DefectVision-AD"

try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
except Exception as exc:
    IN_COLAB = False
    print('Colab이 아닌 환경입니다. Drive mount 생략:', repr(exc))

DRIVE_ROOT = Path('/content/drive/MyDrive') / PROJECT_NAME if IN_COLAB else Path.cwd() / 'drive_sim' / PROJECT_NAME
DATA_ROOT = DRIVE_ROOT / 'data' / 'raw'
MVTEC_ROOT = DATA_ROOT / 'mvtec'
VISA_ROOT = DATA_ROOT / 'visa_mvtec'
OUTPUT_ROOT = DRIVE_ROOT / 'outputs'
RESULT_ROOT = OUTPUT_ROOT / 'multi_category_results'
CHECKPOINT_ROOT = OUTPUT_ROOT / 'checkpoints'
DEPLOY_ROOT = DRIVE_ROOT / 'deploy'

for p in [DRIVE_ROOT, DATA_ROOT, OUTPUT_ROOT, RESULT_ROOT, CHECKPOINT_ROOT, DEPLOY_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

# repo clone 또는 재사용
FORCE_RECLONE = False
if FORCE_RECLONE and PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

if not PROJECT_DIR.exists():
    print('GitHub repo clone:', GITHUB_REPO_URL)
    subprocess.check_call(['git', 'clone', GITHUB_REPO_URL, str(PROJECT_DIR)])
else:
    print('기존 GitHub repo 사용:', PROJECT_DIR)

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# Git 사용자 정보는 commit용. push 인증은 별도 token 환경변수 사용.
subprocess.run(['git', 'config', '--global', 'user.email', GITHUB_EMAIL], check=False)
subprocess.run(['git', 'config', '--global', 'user.name', GITHUB_USERNAME], check=False)

print('IN_COLAB       =', IN_COLAB)
print('PROJECT_DIR    =', PROJECT_DIR)
print('DRIVE_ROOT     =', DRIVE_ROOT)
print('MVTEC_ROOT     =', MVTEC_ROOT)
print('VISA_ROOT      =', VISA_ROOT)
print('RESULT_ROOT    =', RESULT_ROOT)
print('CHECKPOINT_ROOT=', CHECKPOINT_ROOT)

# Colab 기본 패키지 버전 확인. 버전 꼬임 방지를 위해 강제 설치하지 않음.
import importlib
print('\n===== 주요 패키지 버전 =====')
for name in ['numpy', 'pandas', 'torch', 'torchvision', 'cv2', 'sklearn', 'matplotlib', 'PIL', 'yaml', 'tqdm']:
    try:
        mod = importlib.import_module(name)
        ver = getattr(mod, '__version__', 'unknown')
        if name == 'PIL':
            from PIL import Image
            ver = Image.__version__
        print(f'{name:12s}: {ver}')
    except Exception as exc:
        print(f'{name:12s}: IMPORT ERROR -> {repr(exc)}')

# 작은 유틸 패키지만 누락 시 설치. 핵심 ML 패키지는 설치하지 않음.
for import_name, pip_name in [('yaml', 'PyYAML'), ('tqdm', 'tqdm')]:
    try:
        importlib.import_module(import_name)
    except Exception:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])

# torch.load weights_only 기본값 변경에 대비한 안전 패치
# PyTorch 2.6+에서는 torch.load 기본 weights_only가 바뀌어 기존 checkpoint 로드가 실패할 수 있음.
inference_path = PROJECT_DIR / 'src' / 'inference.py'
if inference_path.exists():
    text = inference_path.read_text(encoding='utf-8')
    old = 'checkpoint = torch.load(checkpoint_path, map_location=resolved_device)'
    new = """\n    try:\n        checkpoint = torch.load(checkpoint_path, map_location=resolved_device, weights_only=False)\n    except TypeError:\n        checkpoint = torch.load(checkpoint_path, map_location=resolved_device)\n    """.rstrip()
    if old in text:
        text = text.replace(old, new)
        inference_path.write_text(text, encoding='utf-8')
        print('patched:', inference_path)

print('\n현재 작업 디렉토리:', Path.cwd())
print('src 존재 여부:', (PROJECT_DIR / 'src').exists())

Mounted at /content/drive
GitHub repo clone: https://github.com/wnstjq0915/DefectVision-AD-Proj.git
IN_COLAB       = True
PROJECT_DIR    = /content/DefectVision-AD-Proj
DRIVE_ROOT     = /content/drive/MyDrive/DefectVision-AD
MVTEC_ROOT     = /content/drive/MyDrive/DefectVision-AD/data/raw/mvtec
VISA_ROOT      = /content/drive/MyDrive/DefectVision-AD/data/raw/visa_mvtec
RESULT_ROOT    = /content/drive/MyDrive/DefectVision-AD/outputs/multi_category_results
CHECKPOINT_ROOT= /content/drive/MyDrive/DefectVision-AD/outputs/checkpoints

===== 주요 패키지 버전 =====
numpy       : 2.0.2
pandas      : 2.2.2
torch       : 2.11.0+cpu
torchvision : 0.26.0+cpu
cv2         : 4.13.0
sklearn     : 1.6.1
matplotlib  : 3.10.0
PIL         : 11.3.0
yaml        : 6.0.3
tqdm        : 4.67.3
patched: /content/DefectVision-AD-Proj/src/inference.py

현재 작업 디렉토리: /content/DefectVision-AD-Proj
src 존재 여부: True


In [2]:
# ===== 결과 및 registry 확인 =====
from pathlib import Path
import json
import shutil
import zipfile

required_files = [
    RESULT_ROOT / 'all_results.csv',
    RESULT_ROOT / 'leaderboard.csv',
    RESULT_ROOT / 'best_by_category.csv',
    RESULT_ROOT / 'model_summary.csv',
    RESULT_ROOT / 'category_summary.csv',
    DEPLOY_ROOT / 'model_registry.json',
    DEPLOY_ROOT / 'demo_samples_manifest.json',
]
for p in required_files:
    print(('OK  ' if p.exists() else 'MISS'), p)

missing = [p for p in required_files if not p.exists()]
if missing:
    raise FileNotFoundError('먼저 01 노트북을 실행해 결과를 생성하세요: ' + ', '.join(map(str, missing)))

OK   /content/drive/MyDrive/DefectVision-AD/outputs/multi_category_results/all_results.csv
OK   /content/drive/MyDrive/DefectVision-AD/outputs/multi_category_results/leaderboard.csv
OK   /content/drive/MyDrive/DefectVision-AD/outputs/multi_category_results/best_by_category.csv
OK   /content/drive/MyDrive/DefectVision-AD/outputs/multi_category_results/model_summary.csv
OK   /content/drive/MyDrive/DefectVision-AD/outputs/multi_category_results/category_summary.csv
OK   /content/drive/MyDrive/DefectVision-AD/deploy/model_registry.json
OK   /content/drive/MyDrive/DefectVision-AD/deploy/demo_samples_manifest.json


In [3]:
# ===== Streamlit app 및 EC2 배포 파일 생성 =====
app_dir = PROJECT_DIR / 'app'
scripts_dir = PROJECT_DIR / 'scripts'
streamlit_config_dir = PROJECT_DIR / '.streamlit'
app_dir.mkdir(parents=True, exist_ok=True)
scripts_dir.mkdir(parents=True, exist_ok=True)
streamlit_config_dir.mkdir(parents=True, exist_ok=True)

streamlit_code = r"""

from __future__ import annotations

import json
import os
import random
import sys
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import streamlit as st
from PIL import Image

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RESULT_DIR = ROOT / "docs" / "assets" / "results"
DEFAULT_DEPLOY_ROOT = ROOT / "deploy"
DEPLOY_ROOT = Path(os.environ.get("DEFECTVISION_DEPLOY_ROOT", str(DEFAULT_DEPLOY_ROOT)))
MODEL_ROOT = Path(os.environ.get("DEFECTVISION_MODEL_ROOT", str(DEPLOY_ROOT / "models")))
REGISTRY_PATH = Path(os.environ.get("DEFECTVISION_REGISTRY", str(DEPLOY_ROOT / "model_registry.json")))
SAMPLES_MANIFEST_PATH = Path(os.environ.get("DEFECTVISION_SAMPLES", str(DEPLOY_ROOT / "demo_samples_manifest.json")))
SAMPLES_ROOT = DEPLOY_ROOT

st.set_page_config(
    page_title="DefectVision-AD Demo",
    page_icon="🔎",
    layout="wide",
)


def read_json(path: Path, default: Any):
    if path.exists():
        with path.open("r", encoding="utf-8") as f:
            return json.load(f)
    return default


@st.cache_data(show_spinner=False)
def load_tables():
    tables = {}
    for name in ["all_results", "leaderboard", "best_by_category", "model_summary", "category_summary"]:
        path = RESULT_DIR / f"{name}.csv"
        if path.exists():
            tables[name] = pd.read_csv(path)
        else:
            tables[name] = pd.DataFrame()
    return tables


@st.cache_data(show_spinner=False)
def load_registry_and_samples():
    registry = read_json(REGISTRY_PATH, {"models": []})
    samples = read_json(SAMPLES_MANIFEST_PATH, [])
    return registry, samples


@st.cache_resource(show_spinner=True)
def load_cached_predictor(checkpoint_path: str, image_size: int | None = None):
    from src.inference import load_predictor
    return load_predictor(checkpoint_path, device="auto", image_size=image_size)


def resolve_checkpoint(entry: dict[str, Any]) -> Path:
    rel = entry.get("checkpoint_relative_path")
    if rel:
        # registry stores models/mvtec/category/model.pt; MODEL_ROOT points to deploy/models.
        rel_path = Path(rel)
        if rel_path.parts and rel_path.parts[0] == "models":
            rel_path = Path(*rel_path.parts[1:])
        path = MODEL_ROOT / rel_path
        if path.exists():
            return path
    drive_path = entry.get("checkpoint_drive_path")
    if drive_path and Path(drive_path).exists():
        return Path(drive_path)
    return MODEL_ROOT / "missing.pt"


def model_entries_for_category(registry: dict[str, Any], category: str):
    return [m for m in registry.get("models", []) if m.get("category") == category]


def sample_path(sample: dict[str, Any]) -> Path:
    p = Path(sample.get("sample_path", ""))
    if p.is_absolute():
        return p
    return SAMPLES_ROOT / p


def show_metric_cards(best_df: pd.DataFrame):
    if best_df.empty:
        st.info("아직 best_by_category.csv가 없습니다. Colab 01 노트북을 먼저 실행하세요.")
        return
    cols = st.columns(4)
    cols[0].metric("평가 category 수", len(best_df))
    if "accuracy" in best_df:
        cols[1].metric("평균 Accuracy", f"{pd.to_numeric(best_df['accuracy'], errors='coerce').mean():.3f}")
    if "f1" in best_df:
        cols[2].metric("평균 F1", f"{pd.to_numeric(best_df['f1'], errors='coerce').mean():.3f}")
    if "selection_score" in best_df:
        cols[3].metric("평균 Selection score", f"{pd.to_numeric(best_df['selection_score'], errors='coerce').mean():.3f}")


def display_result_figures():
    figure_files = [
        "fig_model_average_selection_score.png",
        "fig_category_average_accuracy.png",
        "fig_best_model_by_category.png",
        "fig_selection_score_heatmap.png",
        "fig_accuracy_by_category_model.png",
    ]
    existing = [RESULT_DIR / f for f in figure_files if (RESULT_DIR / f).exists()]
    if not existing:
        st.info("README/Streamlit용 결과 그래프가 아직 없습니다. Colab 01, 02 노트북을 실행하세요.")
        return
    for path in existing:
        st.image(str(path), caption=path.name, use_container_width=True)


def predict_and_show(entry: dict[str, Any], image_path: Path):
    checkpoint_path = resolve_checkpoint(entry)
    if not checkpoint_path.exists():
        st.error(
            "선택한 category의 checkpoint를 찾지 못했습니다.\n\n"
            f"찾은 경로: `{checkpoint_path}`\n\n"
            "EC2에서는 deploy bundle을 압축 해제하고 `DEFECTVISION_DEPLOY_ROOT` 또는 `DEFECTVISION_MODEL_ROOT`를 설정해야 합니다."
        )
        return

    image_size = int(entry.get("image_size", 256) or 256)
    predictor = load_cached_predictor(str(checkpoint_path), image_size=image_size)
    pred = predictor.predict_image(image_path)

    from src.visualize import make_heatmap, overlay_heatmap

    score = float(pred["score"])
    threshold = float(pred["threshold"])
    label = pred["label"]
    anomaly_map = pred["anomaly_map"]
    overlay = overlay_heatmap(image_path, anomaly_map)
    heatmap = make_heatmap(anomaly_map)

    cols = st.columns(3)
    cols[0].image(str(image_path), caption="선택 이미지", use_container_width=True)
    cols[1].image(heatmap, caption="Anomaly map", use_container_width=True)
    cols[2].image(overlay, caption="Heatmap overlay", use_container_width=True)

    st.subheader("판정 결과")
    c1, c2, c3 = st.columns(3)
    c1.metric("Prediction", "불량 의심" if label == "anomaly" else "정상")
    c2.metric("Anomaly score", f"{score:.6f}")
    c3.metric("Threshold", f"{threshold:.6f}")


def main():
    st.title("DefectVision-AD: 산업 제품 결함 이상탐지 데모")
    st.caption("MVTec AD category별 모델 성능 비교와 랜덤 test image 추론 데모")

    tables = load_tables()
    registry, samples = load_registry_and_samples()
    best_df = tables.get("best_by_category", pd.DataFrame())
    all_df = tables.get("all_results", pd.DataFrame())

    tab_dashboard, tab_demo, tab_files = st.tabs(["성능 대시보드", "랜덤 이미지 추론", "배포 파일 상태"])

    with tab_dashboard:
        st.header("모델·category별 성능 비교")
        show_metric_cards(best_df)
        if not all_df.empty:
            st.subheader("전체 평가 결과")
            st.dataframe(all_df, use_container_width=True)
        if not best_df.empty:
            st.subheader("category별 최고 모델")
            st.dataframe(best_df, use_container_width=True)
        st.subheader("README용 시각화")
        display_result_figures()

    with tab_demo:
        st.header("학습에 사용하지 않은 test 이미지 랜덤 추론")
        if not registry.get("models"):
            st.warning("model_registry.json이 없습니다. Colab 01/02/03 노트북을 먼저 실행하세요.")
            return

        categories = sorted({m["category"] for m in registry.get("models", [])})
        category = st.selectbox("Category", categories)
        entries = model_entries_for_category(registry, category)
        if not entries:
            st.error("선택한 category의 모델 registry가 없습니다.")
            return

        entry_labels = [f"{e['model']} | score={e.get('selection_score', None)}" for e in entries]
        selected_idx = st.selectbox("사용할 모델", range(len(entries)), format_func=lambda i: entry_labels[i])
        entry = entries[selected_idx]

        category_samples = [s for s in samples if s.get("category") == category]
        if not category_samples:
            st.warning("이 category의 demo sample이 없습니다. Colab 01에서 sample pool을 생성하세요.")
            return

        session_key = f"random_samples_{category}"
        if st.button("랜덤 test 이미지 10개 뽑기", type="primary") or session_key not in st.session_state:
            st.session_state[session_key] = random.sample(category_samples, k=min(10, len(category_samples)))

        selected_samples = st.session_state[session_key]
        labels = [f"{i+1:02d}. {Path(s['sample_path']).name} | 실제={s.get('label_name')} | defect={s.get('defect_type')}" for i, s in enumerate(selected_samples)]
        pick = st.selectbox("이미지 선택", range(len(selected_samples)), format_func=lambda i: labels[i])
        sample = selected_samples[pick]
        img_path = sample_path(sample)

        if not img_path.exists():
            st.error(f"sample image가 없습니다: {img_path}")
            return

        st.info(f"실제 label: {sample.get('label_name')} / defect_type: {sample.get('defect_type')}")
        predict_and_show(entry, img_path)

    with tab_files:
        st.header("배포 파일 상태")
        st.write("Repository root:", ROOT)
        st.write("Result dir:", RESULT_DIR, RESULT_DIR.exists())
        st.write("Deploy root:", DEPLOY_ROOT, DEPLOY_ROOT.exists())
        st.write("Model root:", MODEL_ROOT, MODEL_ROOT.exists())
        st.write("Registry:", REGISTRY_PATH, REGISTRY_PATH.exists())
        st.write("Samples manifest:", SAMPLES_MANIFEST_PATH, SAMPLES_MANIFEST_PATH.exists())
        st.write("모델 수:", len(registry.get("models", [])))
        st.write("sample 수:", len(samples))


if __name__ == "__main__":
    main()

"""
(app_dir / 'streamlit_app.py').write_text(streamlit_code.strip() + '\n', encoding='utf-8')

requirements_ec2 = r"""
# EC2 deployment requirements for DefectVision-AD Streamlit app
# Choose CPU PyTorch wheels unless you use a GPU EC2 instance.
streamlit>=1.36
numpy>=1.26,<2.1
pandas>=2.2
matplotlib>=3.8
Pillow>=10.0
opencv-python-headless>=4.9
scikit-learn>=1.4
PyYAML>=6.0
tqdm>=4.66
torch>=2.2
torchvision>=0.17
"""
(PROJECT_DIR / 'requirements-ec2.txt').write_text(requirements_ec2.strip() + '\n', encoding='utf-8')

config_toml = r"""
[server]
address = "0.0.0.0"
port = 8501
headless = true

[browser]
gatherUsageStats = false
"""
(streamlit_config_dir / 'config.toml').write_text(config_toml.strip() + '\n', encoding='utf-8')

run_script = r"""#!/usr/bin/env bash
set -euo pipefail
export DEFECTVISION_DEPLOY_ROOT="${DEFECTVISION_DEPLOY_ROOT:-$PWD/deploy}"
python -m streamlit run app/streamlit_app.py --server.address=0.0.0.0 --server.port=8501 --server.headless=true
"""
run_path = scripts_dir / 'run_streamlit.sh'
run_path.write_text(run_script, encoding='utf-8')
run_path.chmod(0o755)

service_text = r"""[Unit]
Description=DefectVision-AD Streamlit App
After=network.target

[Service]
User=ubuntu
WorkingDirectory=/home/ubuntu/DefectVision-AD-Proj
Environment=DEFECTVISION_DEPLOY_ROOT=/home/ubuntu/DefectVision-AD-Proj/deploy
ExecStart=/home/ubuntu/DefectVision-AD-Proj/.venv/bin/python -m streamlit run app/streamlit_app.py --server.address=0.0.0.0 --server.port=8501 --server.headless=true
Restart=always
RestartSec=5

[Install]
WantedBy=multi-user.target
"""
(scripts_dir / 'defectvision-streamlit.service').write_text(service_text, encoding='utf-8')

print('wrote Streamlit/EC2 files')

wrote Streamlit/EC2 files


In [4]:
# ===== docs/assets/results로 CSV/PNG 복사 =====
repo_results = PROJECT_DIR / 'docs' / 'assets' / 'results'
repo_results.mkdir(parents=True, exist_ok=True)

for src in RESULT_ROOT.glob('*.csv'):
    shutil.copy2(src, repo_results / src.name)
    print('copied:', src.name)
fig_src = RESULT_ROOT / 'figures'
if fig_src.exists():
    for src in fig_src.glob('*.png'):
        shutil.copy2(src, repo_results / src.name)
        print('copied:', src.name)

copied: mvtec_category_manifest.csv
copied: all_results.csv
copied: autoencoder_history_summary.csv
copied: leaderboard.csv
copied: best_by_category.csv
copied: model_summary.csv
copied: category_summary.csv
copied: fig_model_average_selection_score.png
copied: fig_category_average_accuracy.png
copied: fig_best_model_by_category.png
copied: fig_selection_score_heatmap.png
copied: fig_accuracy_by_category_model.png
copied: fig_autoencoder_best_val_loss_by_category.png
copied: fig_autoencoder_best_epoch_by_category.png


In [5]:
# ===== deploy bundle 생성: best checkpoints + registry + sample images =====
# 이 zip은 GitHub에 올리지 않습니다. EC2로 옮겨서 repo 루트의 deploy/로 압축 해제합니다.

bundle_root = DEPLOY_ROOT / 'bundle_root'
if bundle_root.exists():
    shutil.rmtree(bundle_root)
bundle_root.mkdir(parents=True, exist_ok=True)

# registry와 sample manifest
shutil.copy2(DEPLOY_ROOT / 'model_registry.json', bundle_root / 'model_registry.json')
shutil.copy2(DEPLOY_ROOT / 'demo_samples_manifest.json', bundle_root / 'demo_samples_manifest.json')

# samples 복사
src_samples = DEPLOY_ROOT / 'demo_samples'
dst_samples = bundle_root / 'demo_samples'
if src_samples.exists():
    shutil.copytree(src_samples, dst_samples)

# best checkpoint 복사
registry = json.loads((DEPLOY_ROOT / 'model_registry.json').read_text(encoding='utf-8'))
for entry in registry.get('models', []):
    src = Path(entry['checkpoint_drive_path'])
    dst_rel = Path(entry['checkpoint_relative_path'])
    # bundle_root에는 deploy 내부 기준으로 models/... 형태가 들어가야 함
    dst = bundle_root / dst_rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.exists():
        shutil.copy2(src, dst)
        print('model copied:', src, '->', dst)
    else:
        print('[WARN] checkpoint missing:', src)

# 결과 csv/png도 bundle에 포함
bundle_results = bundle_root / 'results'
bundle_results.mkdir(parents=True, exist_ok=True)
for src in repo_results.glob('*'):
    if src.is_file():
        shutil.copy2(src, bundle_results / src.name)

zip_path = DEPLOY_ROOT / 'defectvision_ec2_deploy_bundle.zip'
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in bundle_root.rglob('*'):
        if p.is_file():
            zf.write(p, p.relative_to(bundle_root))

print('deploy bundle:', zip_path)
print('size MB:', zip_path.stat().st_size / 1024 / 1024)

model copied: /content/drive/MyDrive/DefectVision-AD/outputs/checkpoints/mvtec/bottle/patchcore.pt -> /content/drive/MyDrive/DefectVision-AD/deploy/bundle_root/models/mvtec/bottle/patchcore.pt
model copied: /content/drive/MyDrive/DefectVision-AD/outputs/checkpoints/mvtec/cable/patchcore.pt -> /content/drive/MyDrive/DefectVision-AD/deploy/bundle_root/models/mvtec/cable/patchcore.pt
model copied: /content/drive/MyDrive/DefectVision-AD/outputs/checkpoints/mvtec/capsule/patchcore.pt -> /content/drive/MyDrive/DefectVision-AD/deploy/bundle_root/models/mvtec/capsule/patchcore.pt
model copied: /content/drive/MyDrive/DefectVision-AD/outputs/checkpoints/mvtec/carpet/patchcore.pt -> /content/drive/MyDrive/DefectVision-AD/deploy/bundle_root/models/mvtec/carpet/patchcore.pt
model copied: /content/drive/MyDrive/DefectVision-AD/outputs/checkpoints/mvtec/grid/patchcore.pt -> /content/drive/MyDrive/DefectVision-AD/deploy/bundle_root/models/mvtec/grid/patchcore.pt
model copied: /content/drive/MyDrive/De

In [6]:
# ===== README 결과 섹션 삽입/갱신 =====
import pandas as pd

readme_path = PROJECT_DIR / 'README.md'
text = readme_path.read_text(encoding='utf-8')

best = pd.read_csv(RESULT_ROOT / 'best_by_category.csv')
model_summary = pd.read_csv(RESULT_ROOT / 'model_summary.csv')

best_cols = [c for c in ['category', 'model', 'accuracy', 'f1', 'auroc', 'pixel_auroc', 'selection_score'] if c in best.columns]
model_cols = [c for c in ['model', 'accuracy', 'f1', 'auroc', 'pixel_auroc', 'selection_score'] if c in model_summary.columns]

section = f"""
<!-- DEFECTVISION_RESULTS_START -->

## 실험 결과: MVTec 전체 category 모델 비교

이 섹션은 Colab 노트북으로 자동 생성되었습니다. 전체 category별로 AutoEncoder/PatchCore를 학습·평가하고, 단순 Accuracy뿐 아니라 F1, AUROC, Pixel AUROC를 함께 고려한 `selection_score`로 category별 대표 모델을 선택했습니다.

### 모델 선택 기준

```text
selection_score = 0.40 * AUROC + 0.25 * F1 + 0.25 * Pixel_AUROC + 0.10 * Accuracy
```

### 모델 평균 성능

{model_summary[model_cols].to_markdown(index=False)}

### Category별 최고 모델

{best[best_cols].to_markdown(index=False)}

### 시각화

![Model average selection score](docs/assets/results/fig_model_average_selection_score.png)

![Category average accuracy](docs/assets/results/fig_category_average_accuracy.png)

![Best model by category](docs/assets/results/fig_best_model_by_category.png)

![Selection score heatmap](docs/assets/results/fig_selection_score_heatmap.png)

![Accuracy by category and model](docs/assets/results/fig_accuracy_by_category_model.png)

### Streamlit 데모

EC2 배포 후 사용자는 웹사이트에서 category를 선택하고, 학습에 사용되지 않은 test image 후보 중 랜덤 10개를 뽑아 하나를 선택한 뒤 정상/불량 판정과 heatmap overlay를 확인할 수 있습니다.

<!-- DEFECTVISION_RESULTS_END -->
"""

start = '<!-- DEFECTVISION_RESULTS_START -->'
end = '<!-- DEFECTVISION_RESULTS_END -->'
if start in text and end in text:
    before = text.split(start)[0]
    after = text.split(end)[1]
    text = before + section + after
else:
    text = text.rstrip() + '\n\n' + section + '\n'

readme_path.write_text(text, encoding='utf-8')
print('updated:', readme_path)

updated: /content/DefectVision-AD-Proj/README.md


In [7]:
# ===== GitHub push: 환경변수 DEFECTVISION_GH_TOKEN 사용 =====
# Fine-grained PAT 권장:
# - Repository access: wnstjq0915/DefectVision-AD-Proj
# - Contents: Read and write

import os
import stat
from getpass import getpass

TOKEN_ENV_NAME = 'DEFECTVISION_GH_TOKEN'
if TOKEN_ENV_NAME not in os.environ or not os.environ[TOKEN_ENV_NAME].strip():
    os.environ[TOKEN_ENV_NAME] = getpass('GitHub Personal Access Token 입력: ')
    print(f'{TOKEN_ENV_NAME} 설정 완료')
else:
    print(f'{TOKEN_ENV_NAME} 이미 설정됨')

os.chdir(PROJECT_DIR)
subprocess.check_call(['git', 'remote', 'set-url', 'origin', GITHUB_REPO_URL])
subprocess.check_call(['git', 'add', 'README.md', 'docs/assets/results', 'app/streamlit_app.py', 'requirements-ec2.txt', '.streamlit/config.toml', 'scripts'])

# 변경사항 있을 때만 commit
if subprocess.run(['git', 'diff', '--cached', '--quiet']).returncode == 0:
    print('커밋할 변경사항이 없습니다.')
else:
    subprocess.check_call(['git', 'commit', '-m', 'Add multi-category results and Streamlit demo'])

branch = subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip() or 'main'
print('push branch:', branch)

askpass_path = Path('/content/git_askpass_defectvision.sh')
askpass_path.write_text("""#!/bin/sh
case \"$1\" in
  *Username*) echo \"wnstjq0915\" ;;
  *Password*) echo \"$DEFECTVISION_GH_TOKEN\" ;;
  *) echo \"$DEFECTVISION_GH_TOKEN\" ;;
esac
""", encoding='utf-8')
askpass_path.chmod(askpass_path.stat().st_mode | stat.S_IXUSR)

env = os.environ.copy()
env['GIT_ASKPASS'] = str(askpass_path)
env['GIT_TERMINAL_PROMPT'] = '0'

subprocess.check_call(['git', 'push', 'origin', branch], env=env)
print('GitHub push 완료')

GitHub Personal Access Token 입력: ··········
DEFECTVISION_GH_TOKEN 설정 완료
push branch: main
GitHub push 완료
